# ema-first-moment composite — cx22: track m EMA alongside the sqrt(...)+eps denominator pattern

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `ema-first-moment`, `sqrt-eps-stabilize`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "ema-first-moment"
DD_ATOM_IDS = ["ema-first-moment", "sqrt-eps-stabilize"]
DD_SUBTOPICS = ["Optimizer: Adam EMA first moment", "Numerical: sqrt-eps stabilization"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

This is a **rare-pair** drill: the two atoms don't share a single equation in Adam's step (the canonical denominator uses `v_hat`, not `m`), but they DO co-locate in Adam-from-scratch implementations — both live inside the per-parameter inner loop.

**The pair.**
- **ema-first-moment** — `m = beta1 * m + (1 - beta1) * g`. The numerator buffer.
- **sqrt-eps-stabilize** — `denom = sqrt(positive_buffer) + eps` or equivalently `sqrt(positive_buffer + eps)`. The defensive stabilization.

**Why pair them.** In ARENA's Adam writeup the inner loop interleaves: update m, update v, build the sqrt-eps denominator from v (or from an arbitrary non-negative buffer), form the step. The most common writing bug is to use the WRONG buffer in the sqrt-eps denominator (e.g. `sqrt(m**2) + eps` — a sign-discarding error) or to apply eps in the wrong place. This drill exercises the safe composition: m is updated, and a non-negative scratch buffer (passed in) gets the sqrt-eps treatment, returning both.

**Where to put the eps.** ARENA / `torch.optim.Adam` use `sqrt(v_hat) + eps` (eps OUTSIDE the sqrt). Some BatchNorm/RMSNorm impls use `sqrt(var + eps)` (eps INSIDE). Both stabilize the divide; only the inside-sqrt form bounds the gradient near zero. Adam tolerates the outside-sqrt form because its `v_hat` is bounded away from zero by the warmup-step accumulation. For this drill, use the **Adam convention**: `sqrt(buf) + eps`.

### Composite Exercise — track m EMA alongside the sqrt(...)+eps denominator pattern

**Atoms exercised together**: `ema-first-moment`, `sqrt-eps-stabilize`

Implement `cx22_m_step_and_denom(m, g, buf, beta1, eps)`.

Inputs:
- `m`: first-moment buffer (Tensor).
- `g`: gradient (Tensor, same shape as `m`).
- `buf`: a non-negative scratch buffer (Tensor) used to build the sqrt-eps denominator. Assume `buf >= 0` elementwise — the caller is responsible.
- `beta1`: float decay.
- `eps`: float stabilizer (typically 1e-8).

Returns `(m_new, denom)`:
- `m_new = beta1 * m + (1 - beta1) * g` (atom: ema-first-moment).
- `denom = sqrt(buf) + eps` — Adam convention, eps OUTSIDE the sqrt (atom: sqrt-eps-stabilize).

Do not mutate `m` or `buf`.

Tests verify:
- `m_new` follows the EMA closed form.
- `denom > 0` strictly, even when `buf == 0`.
- `denom` matches `sqrt(buf) + eps` to numerical precision.
- The OUTSIDE-sqrt placement is detectable: when `buf` is all-zero, `denom == eps`, not `sqrt(eps) ~ 3.16e-4`.
- The two atoms are INDEPENDENT — changing `buf` does not affect `m_new`, and changing `g` does not affect `denom`.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx22_m_step_and_denom(m: Tensor, g: Tensor, buf: Tensor, beta1: float, eps: float):
    """Return (m_new, denom) — EMA step on m plus sqrt-eps denominator from buf."""
    raise NotImplementedError

def _test_cx22():
    # Case A: m EMA step matches the recurrence.
    m = t.zeros(4)
    g = t.tensor([1.0, -2.0, 3.0, 0.5])
    buf = t.tensor([1.0, 4.0, 9.0, 16.0])
    beta1 = 0.9
    eps = 1e-8
    m_new, denom = cx22_m_step_and_denom(m, g, buf, beta1, eps)
    assert t.allclose(m_new, 0.1 * g, atol=1e-6), f'm_new wrong: got {m_new}'

    # Case B: denom matches sqrt(buf) + eps.
    expected_denom = buf.sqrt() + eps
    assert t.allclose(denom, expected_denom, atol=1e-7), (
        f'denom wrong: got {denom}, expected {expected_denom}'
    )

    # Case C: denom > 0 strictly, even on all-zero buf.
    buf_zero = t.zeros(3)
    m2 = t.zeros(3)
    g2 = t.ones(3)
    _, denom_zero = cx22_m_step_and_denom(m2, g2, buf_zero, 0.9, eps=1e-8)
    assert (denom_zero > 0).all(), f'denom must be strictly positive; got {denom_zero}'
    # Adam convention: denom == eps when buf == 0 (NOT sqrt(eps)).
    assert t.allclose(denom_zero, t.full((3,), 1e-8), atol=1e-12), (
        f'on buf=0 with eps OUTSIDE sqrt, denom should be eps={1e-8}; got {denom_zero} — '
        f'if you got ~3.16e-4 you put eps INSIDE the sqrt (sqrt(buf+eps)), use Adam convention.'
    )

    # Case D: inputs not mutated.
    m_in = t.tensor([0.5, 1.0])
    buf_in = t.tensor([1.0, 2.0])
    snap_m, snap_buf = m_in.clone(), buf_in.clone()
    _ = cx22_m_step_and_denom(m_in, t.zeros(2), buf_in, 0.9, eps=1e-8)
    assert t.equal(m_in, snap_m), 'm was mutated'
    assert t.equal(buf_in, snap_buf), 'buf was mutated'

    # Case E: independence — changing buf doesn't change m_new.
    m_a, _ = cx22_m_step_and_denom(t.zeros(3), t.ones(3), t.tensor([1.0, 2.0, 3.0]), 0.9, 1e-8)
    m_b, _ = cx22_m_step_and_denom(t.zeros(3), t.ones(3), t.tensor([99.0, 88.0, 77.0]), 0.9, 1e-8)
    assert t.allclose(m_a, m_b), 'm_new must not depend on buf'

    # Case F: independence — changing g doesn't change denom.
    _, d_a = cx22_m_step_and_denom(t.zeros(3), t.tensor([1.0, 2.0, 3.0]), t.ones(3), 0.9, 1e-8)
    _, d_b = cx22_m_step_and_denom(t.zeros(3), t.tensor([99.0, 88.0, 77.0]), t.ones(3), 0.9, 1e-8)
    assert t.allclose(d_a, d_b), 'denom must not depend on g'

    # Case G: many-step EMA of m converges to g (atom-A sanity).
    m = t.zeros(2)
    g = t.tensor([1.0, -2.0])
    for _ in range(500):
        m, _ = cx22_m_step_and_denom(m, g, t.ones(2), 0.9, 1e-8)
    assert t.allclose(m, g, atol=1e-4), f'm should converge to g for constant g; got {m}'
    _dd_passed.add('cx22')

_test_cx22()

<details><summary>Show solution — cx22</summary>

```python
def cx22_m_step_and_denom(m, g, buf, beta1, eps):
    # Atom A (ema-first-moment): first-moment EMA step.
    m_new = beta1 * m + (1.0 - beta1) * g
    # Atom B (sqrt-eps-stabilize): Adam convention puts eps OUTSIDE the sqrt.
    denom = buf.sqrt() + eps
    return m_new, denom
```

**Eps placement matters.** Adam uses `sqrt(v_hat) + eps` (eps outside) because `v_hat` stays bounded away from zero — bias correction at step 1 already inflates `v` by `1/(1-beta2)` ~ 1000x. BatchNorm uses `sqrt(var + eps)` (eps inside) because batch var CAN collapse to zero on a dead-ReLU channel.

**Test discrimination.** Case C is the cleanest way to tell the two placements apart: with `buf == 0`, eps-outside gives `eps = 1e-8`, eps-inside gives `sqrt(1e-8) = 1e-4` — a 10,000x difference.

**Why this rare pair is worth exercising.** ARENA's Adam-from-scratch lab interleaves all four atoms (m, v, bias correction, sqrt-eps denominator) inside one per-parameter loop. Pinning down the m + sqrt-eps composition independently helps catch wrong-buffer bugs (e.g. `sqrt(m_squared) + eps`) that pass the m-only and sqrt-eps-only tests.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx22'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx22',
        'subtopics': ["Optimizer: Adam EMA first moment", "Numerical: sqrt-eps stabilization"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()